# PERSUADE: Local Lag Curve Analysis (Sliding Windows)

## What We're Measuring

For each window, we measure **NLL on the true next tokens** (the actual words the student wrote), not distributional "fluency" in the abstract.

For each lag k, we compute:
```
gain_k = NLL(k=0) − NLL(k)
```
i.e., **how much the immediately preceding k tokens help predict the next 32 tokens**.

The shape of `gain_k` as k increases is the **"local memory curve"**: how quickly predictive benefit saturates.

## Why Sliding Windows Are Essential

1. **Avoids end-of-essay effects** (conclusion phrases, "in summary," etc.)
2. **Provides within-essay reliability**: if the curve is a stable property of a writer/essay, it should look similar across windows
3. **Controls for sampling location**: tests whether group differences are real vs. artifacts of where we sampled

## Primary Outputs: Shape Metrics (not raw gain levels)

**Shape metrics are the main outputs** - they characterize the *memory-use profile* independent of overall fluency:

1. **`early_ratio`** = gain_16 / gain_128
   - What fraction of total benefit comes from the first 16 tokens?
   - Higher = faster saturation

2. **`log_slope_local`** = slope of gain vs log(k)
   - How steeply does benefit increase with context?

3. **`auc_log_k`** (primary) = area under gain curve on log(k) scale
   - Emphasizes early saturation / short-range dynamics
   
4. **`auc_linear_k`** (secondary) = area under gain curve on linear k scale
   - Emphasizes longer-range contributions

Raw gain levels (e.g., `mean_gain_128`) track fluency/length; the **shape** is what we're treating as the distinctive "memory-use profile."

## Cohort
persuade_score_long_cohort.jsonl (50/50/50 balanced, all long enough)

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes pandas numpy matplotlib seaborn tqdm statsmodels

In [ ]:
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
import statsmodels.formula.api as smf

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Paths
DRIVE_BASE = "/content/drive/MyDrive/LRTIA/Data/persuade_clean"
OUTPUT_BASE = "/content/drive/MyDrive/LRTIA/Results/Persuade"
COHORT_PATH = f"{DRIVE_BASE}/cohorts/persuade_score_long_cohort.jsonl"

EXPERIMENT = 'local_lag_curve_sliding_v1'

# Model
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True  # True for T4, False for A100

# Sliding window parameters
TARGET_LENGTH = 32  # T=32 tokens per window
N_WINDOWS = 20      # Windows per essay
MIN_CONTEXT = 128   # Min tokens of preceding context
END_BUFFER = 64     # Exclude last 64 tokens

# Context length grid (k values)
K_GRID = [0, 2, 4, 8, 12, 16, 24, 32, 48, 64, 96, 128]

# Minimum tokens required
MIN_TOKENS_REQUIRED = MIN_CONTEXT + TARGET_LENGTH + END_BUFFER

# Reproducibility
RANDOM_SEED = 42

# Split-half reliability iterations
N_SPLIT_HALF_ITERS = 20

print(f"Experiment: {EXPERIMENT}")
print(f"Cohort: {COHORT_PATH}")
print(f"\nSliding window parameters:")
print(f"  Target length T: {TARGET_LENGTH} tokens")
print(f"  Windows per essay: {N_WINDOWS}")
print(f"  Min preceding context: {MIN_CONTEXT} tokens")
print(f"  End buffer (excluded): {END_BUFFER} tokens")
print(f"\nContext grid k: {K_GRID}")
print(f"Min tokens required: {MIN_TOKENS_REQUIRED}")
print(f"Random seed: {RANDOM_SEED}")

## 1. Load Model and Data

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load model
print(f"Loading model: {MODEL_NAME}")

if USE_4BIT:
    print("  Using 4-bit quantization")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    print("  Using float16")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )

model.eval()
print("Model loaded")

In [ ]:
# Load cohort
def load_cohort(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

cohort = load_cohort(COHORT_PATH)
print(f"Loaded {len(cohort)} essays")

df_cohort = pd.DataFrame(cohort)
print(f"\nScore bin distribution (original cohort):")
print(df_cohort['score_bin'].value_counts())

In [ ]:
# Pre-tokenize and filter
essay_tokens = {}
excluded_ids = []

for essay in cohort:
    token_ids = tokenizer.encode(essay['text'], add_special_tokens=False)
    if len(token_ids) >= MIN_TOKENS_REQUIRED:
        essay_tokens[essay['essay_id']] = token_ids
    else:
        excluded_ids.append(essay['essay_id'])

print(f"Tokenized essays: {len(essay_tokens)}")
print(f"Excluded (too short): {len(excluded_ids)}")

lengths = [len(t) for t in essay_tokens.values()]
print(f"Token lengths: min={min(lengths)}, median={np.median(lengths):.0f}, max={max(lengths)}")

In [ ]:
# SANITY CHECK: Group selection effects from filtering
print("="*80)
print("SANITY CHECK: Does filtering create group selection bias?")
print("="*80)

included_ids = set(essay_tokens.keys())
df_cohort['included'] = df_cohort['essay_id'].isin(included_ids)

print("\nInclusion by score_bin:")
crosstab = pd.crosstab(df_cohort['score_bin'], df_cohort['included'], margins=True)
print(crosstab)

print("\nInclusion rates:")
for sb in ['low', 'mid', 'high']:
    subset = df_cohort[df_cohort['score_bin'] == sb]
    rate = subset['included'].mean() * 100
    print(f"  {sb}: {rate:.1f}% included")

# Chi-square test for independence
contingency = pd.crosstab(df_cohort['score_bin'], df_cohort['included'])
chi2, p, dof, expected = stats.chi2_contingency(contingency)
print(f"\nChi-square test: χ²={chi2:.2f}, p={p:.4f}")
if p < 0.05:
    print("  WARNING: Significant group selection effect!")
else:
    print("  OK: No significant group selection effect.")

## 2. Core Functions

In [ ]:
@torch.no_grad()
def compute_nll_for_target(context_ids, target_ids):
    """
    Compute mean NLL for predicting target_ids (the actual words written)
    given context_ids.
    """
    full_ids = context_ids + target_ids
    
    if len(context_ids) == 0:
        input_ids = torch.tensor([full_ids], device=model.device)
        outputs = model(input_ids)
        logits = outputs.logits[0]
        
        nlls = []
        for i in range(len(full_ids) - 1):
            log_probs = torch.log_softmax(logits[i], dim=-1)
            true_token = full_ids[i + 1]
            nll = -log_probs[true_token].item()
            nlls.append(nll)
        
        return np.mean(nlls) if nlls else 0.0
    else:
        input_ids = torch.tensor([full_ids], device=model.device)
        outputs = model(input_ids)
        logits = outputs.logits[0]
        
        target_start = len(context_ids)
        nlls = []
        
        # First target token predicted from last context token
        log_probs = torch.log_softmax(logits[target_start - 1], dim=-1)
        nll_first = -log_probs[target_ids[0]].item()
        nlls.append(nll_first)
        
        # Remaining target tokens
        for i in range(len(target_ids) - 1):
            pos = target_start + i
            if pos >= logits.shape[0]:
                break
            log_probs = torch.log_softmax(logits[pos], dim=-1)
            true_token = target_ids[i + 1]
            nll = -log_probs[true_token].item()
            nlls.append(nll)
        
        return np.mean(nlls) if nlls else 0.0


def sample_window_positions(n_tokens, n_windows, rng):
    """
    Sample target window start positions uniformly with small jitter.
    Valid range: [MIN_CONTEXT, n_tokens - TARGET_LENGTH - END_BUFFER)
    """
    min_start = MIN_CONTEXT
    max_start = n_tokens - TARGET_LENGTH - END_BUFFER
    
    if max_start <= min_start:
        return []
    
    if n_windows == 1:
        return [(min_start + max_start) // 2]
    
    positions = np.linspace(min_start, max_start, n_windows, dtype=int).tolist()
    
    jittered = []
    for pos in positions:
        jitter = rng.randint(-5, 5)
        new_pos = max(min_start, min(max_start, pos + jitter))
        jittered.append(new_pos)
    
    return jittered


def run_lag_curve_for_window(token_ids, target_start):
    """
    Compute NLL and gain for each k for a single target window.
    """
    target_end = target_start + TARGET_LENGTH
    target_ids = token_ids[target_start:target_end]
    context_pool = token_ids[:target_start]
    
    nll_by_k = {}
    
    for k in K_GRID:
        if k == 0:
            context_ids = []
        else:
            context_ids = context_pool[-k:] if k <= len(context_pool) else context_pool
        
        nll = compute_nll_for_target(context_ids, target_ids)
        nll_by_k[k] = nll
    
    nll_0 = nll_by_k[0]
    gain_by_k = {k: nll_0 - nll_by_k[k] for k in K_GRID}
    
    return {'nll': nll_by_k, 'gain': gain_by_k}


def compute_auc_log_k(gains_by_k):
    """
    Compute area under gain curve using trapezoidal rule on LOG scale.
    Emphasizes early saturation / short-range dynamics.
    """
    auc = 0
    for i in range(len(K_GRID) - 1):
        k1, k2 = K_GRID[i], K_GRID[i + 1]
        g1, g2 = gains_by_k[k1], gains_by_k[k2]
        width = np.log(k2 + 1) - np.log(k1 + 1)
        auc += 0.5 * (g1 + g2) * width
    return auc


def compute_auc_linear_k(gains_by_k):
    """
    Compute area under gain curve using trapezoidal rule on LINEAR scale.
    Emphasizes longer-range contributions.
    """
    auc = 0
    for i in range(len(K_GRID) - 1):
        k1, k2 = K_GRID[i], K_GRID[i + 1]
        g1, g2 = gains_by_k[k1], gains_by_k[k2]
        width = k2 - k1  # linear width
        auc += 0.5 * (g1 + g2) * width
    return auc


def compute_shape_metrics_from_gains(mean_gains):
    """
    Compute shape metrics from a dict of mean gains by k.
    
    Shape metrics are the PRIMARY outputs - they characterize memory-use profile
    independent of overall fluency.
    """
    results = {}
    
    # early_ratio: What fraction of total benefit comes from first 16 tokens?
    if mean_gains[128] > 0:
        results['early_ratio'] = mean_gains[16] / mean_gains[128]
    else:
        results['early_ratio'] = np.nan
    
    # log_slope_local: How steeply does benefit increase with context?
    ks_for_fit = [k for k in K_GRID if k >= 2]
    log_ks = [np.log(k + 1) for k in ks_for_fit]
    gains_for_fit = [mean_gains[k] for k in ks_for_fit]
    
    if len(log_ks) >= 2:
        slope, intercept, r_value, p_value, std_err = stats.linregress(log_ks, gains_for_fit)
        results['log_slope_local'] = slope
        results['log_slope_r2'] = r_value ** 2
    else:
        results['log_slope_local'] = np.nan
        results['log_slope_r2'] = np.nan
    
    # Dual AUC metrics
    results['auc_log_k'] = compute_auc_log_k(mean_gains)      # PRIMARY: emphasizes early saturation
    results['auc_linear_k'] = compute_auc_linear_k(mean_gains)  # Secondary: emphasizes longer k
    
    # Legacy alias for compatibility
    results['auc_local'] = results['auc_log_k']
    
    return results


def run_sliding_lag_curve_for_essay(token_ids, essay_id, rng):
    """
    Run sliding window lag curve analysis for a single essay.
    Returns essay_results (aggregated) and window_results (raw).
    """
    n_tokens = len(token_ids)
    positions = sample_window_positions(n_tokens, N_WINDOWS, rng)
    
    if len(positions) == 0:
        return None, []
    
    window_results = []
    all_gains = {k: [] for k in K_GRID}
    all_nlls = {k: [] for k in K_GRID}
    window_early_ratios = []
    
    for win_idx, target_start in enumerate(positions):
        win_result = run_lag_curve_for_window(token_ids, target_start)
        
        for k in K_GRID:
            window_results.append({
                'essay_id': essay_id,
                'window_idx': win_idx,
                'target_start': target_start,
                'k': k,
                'nll_k': win_result['nll'][k],
                'gain_k': win_result['gain'][k],
            })
            all_gains[k].append(win_result['gain'][k])
            all_nlls[k].append(win_result['nll'][k])
        
        # Per-window early_ratio
        g16 = win_result['gain'][16]
        g128 = win_result['gain'][128]
        if g128 > 0:
            window_early_ratios.append(g16 / g128)
    
    # Aggregate across windows
    essay_results = {'essay_id': essay_id, 'n_windows': len(positions)}
    
    mean_gains = {}
    for k in K_GRID:
        essay_results[f'mean_nll_{k}'] = np.mean(all_nlls[k])
        essay_results[f'mean_gain_{k}'] = np.mean(all_gains[k])
        essay_results[f'std_gain_{k}'] = np.std(all_gains[k])
        mean_gains[k] = essay_results[f'mean_gain_{k}']
    
    # Shape metrics from aggregated curve (PRIMARY OUTPUTS)
    shape_metrics = compute_shape_metrics_from_gains(mean_gains)
    essay_results.update(shape_metrics)
    
    # Stability metrics
    essay_results['stability_gain_128'] = np.std(all_gains[128])
    if window_early_ratios:
        essay_results['stability_early_ratio'] = np.std(window_early_ratios)
    else:
        essay_results['stability_early_ratio'] = np.nan
    
    # Store window gains for split-half reliability
    essay_results['_window_gains'] = all_gains
    
    # Baseline NLL (for controlling fluency)
    essay_results['baseline_nll'] = essay_results['mean_nll_128']
    
    return essay_results, window_results


print("Core functions defined")

## 3. Run Sliding Window Analysis

In [ ]:
# Build essay metadata lookup
essay_meta = {e['essay_id']: e for e in cohort}

# Initialize RNG
rng = random.Random(RANDOM_SEED)

# Process all essays
all_essay_results = []
all_window_results = []
start_time = time.time()

for essay_id, token_ids in tqdm(essay_tokens.items(), desc="Processing essays"):
    essay_results, window_results = run_sliding_lag_curve_for_essay(token_ids, essay_id, rng)
    
    if essay_results is None:
        continue
    
    # Add metadata
    meta = essay_meta[essay_id]
    essay_results['score'] = meta.get('score')
    essay_results['score_bin'] = meta.get('score_bin')
    essay_results['grade'] = meta.get('grade')
    essay_results['token_count'] = len(token_ids)
    
    all_essay_results.append(essay_results)
    all_window_results.extend(window_results)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f}s ({elapsed/len(essay_tokens):.2f}s/essay)")
print(f"Essays processed: {len(all_essay_results)}")
print(f"Total windows: {len(all_window_results) // len(K_GRID)}")

In [ ]:
# Create DataFrames
df = pd.DataFrame(all_essay_results)
df_windows = pd.DataFrame(all_window_results)

# Remove internal column
window_gains_by_essay = {r['essay_id']: r['_window_gains'] for r in all_essay_results}
df = df.drop(columns=['_window_gains'])

print(f"Essay-level results: {df.shape}")
print(f"Window-level results: {df_windows.shape}")

df.head()

## 4. Sanity Checks

In [ ]:
print("="*80)
print("SANITY CHECK 1: Monotonicity of gain_k")
print("="*80)
print("\ngain_k should generally be monotone increasing with k.")
print("Non-monotonicity is expected occasionally due to noise; we report its prevalence.\n")

# Aggregate level check
prev_gain = None
agg_monotone_violations = []

print(f"{'k':>5} {'mean_gain':>12} {'std':>10} {'N_windows':>12} {'vs prev':>10}")
print("-"*55)

for k in K_GRID:
    col = f'mean_gain_{k}'
    mean_val = df[col].mean()
    std_val = df[col].std()
    n_windows = len(df_windows[df_windows['k'] == k])
    
    if prev_gain is not None:
        if mean_val < prev_gain:
            mono_status = "↓"
            agg_monotone_violations.append(k)
        else:
            mono_status = "↑"
    else:
        mono_status = "-"
    
    print(f"{k:>5} {mean_val:>12.4f} {std_val:>10.4f} {n_windows:>12} {mono_status:>10}")
    prev_gain = mean_val

print(f"\nAggregate (mean across essays): {'Monotone' if not agg_monotone_violations else f'Violations at k={agg_monotone_violations}'}")

# Per-essay monotonicity analysis
print("\n" + "-"*55)
print("Per-Essay Monotonicity Analysis:")
print("-"*55)

essays_with_violations = 0
total_violations = 0
violation_magnitudes = []

for _, row in df.iterrows():
    essay_violations = 0
    prev_g = None
    for k in K_GRID:
        g = row[f'mean_gain_{k}']
        if prev_g is not None and g < prev_g:
            essay_violations += 1
            violation_magnitudes.append(prev_g - g)  # magnitude of decrease
        prev_g = g
    
    if essay_violations > 0:
        essays_with_violations += 1
        total_violations += essay_violations

pct_essays_with_violations = 100 * essays_with_violations / len(df)
avg_magnitude = np.mean(violation_magnitudes) if violation_magnitudes else 0

print(f"\n  Essays with any monotonicity violation: {essays_with_violations}/{len(df)} ({pct_essays_with_violations:.1f}%)")
print(f"  Total violations across all essays: {total_violations}")
print(f"  Average violation magnitude: {avg_magnitude:.4f} (decrease in gain)")

if violation_magnitudes:
    print(f"  Max violation magnitude: {max(violation_magnitudes):.4f}")
    print(f"  Median violation magnitude: {np.median(violation_magnitudes):.4f}")

print("\nInterpretation:")
if pct_essays_with_violations < 10:
    print("  Low violation rate (<10%) - curves are reliably monotonic.")
elif pct_essays_with_violations < 25:
    print("  Moderate violation rate (10-25%) - some noise but overall trend is monotonic.")
else:
    print("  High violation rate (>25%) - consider whether k grid or window size is appropriate.")

In [ ]:
print("\n" + "="*80)
print("SANITY CHECK 2: Shape Metrics Distributions")
print("="*80)
print("\nShape metrics are the PRIMARY outputs (memory-use profile):")

for metric in ['early_ratio', 'log_slope_local', 'auc_log_k', 'auc_linear_k']:
    data = df[metric].dropna()
    print(f"\n{metric}:")
    print(f"  N valid: {len(data)} / {len(df)}")
    print(f"  mean={data.mean():.4f}, median={data.median():.4f}, std={data.std():.4f}")
    print(f"  range=[{data.min():.4f}, {data.max():.4f}]")

print("\n" + "-"*55)
print("Secondary metrics (raw gain levels - track fluency/length):")

for metric in ['mean_gain_128', 'baseline_nll']:
    data = df[metric].dropna()
    print(f"\n{metric}:")
    print(f"  N valid: {len(data)} / {len(df)}")
    print(f"  mean={data.mean():.4f}, median={data.median():.4f}, std={data.std():.4f}")

In [ ]:
print("\n" + "="*80)
print("SANITY CHECK 3: Within-Essay Stability")
print("="*80)

print(f"\nstability_gain_128 (SD of gain_128 across windows):")
print(f"  mean={df['stability_gain_128'].mean():.4f}")

print(f"\nstability_early_ratio (SD of early_ratio across windows):")
print(f"  mean={df['stability_early_ratio'].mean():.4f}")

between_essay_sd = df['early_ratio'].std()
mean_within_essay_sd = df['stability_early_ratio'].mean()
print(f"\nBetween-essay SD of early_ratio: {between_essay_sd:.4f}")
print(f"Mean within-essay SD of early_ratio: {mean_within_essay_sd:.4f}")
print(f"Ratio (between/within): {between_essay_sd/mean_within_essay_sd:.2f}x")
print("  (>1 means there's real between-essay signal)")

## 5. Split-Half Reliability

In [ ]:
print("="*80)
print(f"SPLIT-HALF RELIABILITY ({N_SPLIT_HALF_ITERS} iterations)")
print("="*80)
print("\nRandomly split 20 windows into two sets, compute metrics on each half,")
print("report correlation across essays.")

rng_split = random.Random(RANDOM_SEED + 1)

# Primary shape metrics first, then secondary
split_half_correlations = {
    'early_ratio': [], 
    'log_slope_local': [], 
    'auc_log_k': [],
    'auc_linear_k': [],
    'mean_gain_128': []  # secondary
}

for iteration in range(N_SPLIT_HALF_ITERS):
    half1_metrics = []
    half2_metrics = []
    
    for essay_id in df['essay_id']:
        if essay_id not in window_gains_by_essay:
            continue
        
        all_gains = window_gains_by_essay[essay_id]
        n_windows = len(all_gains[0])
        
        if n_windows < 4:  # Need at least 2 per half
            continue
        
        # Random split
        indices = list(range(n_windows))
        rng_split.shuffle(indices)
        half1_idx = indices[:n_windows//2]
        half2_idx = indices[n_windows//2:]
        
        # Compute mean gains for each half
        half1_gains = {k: np.mean([all_gains[k][i] for i in half1_idx]) for k in K_GRID}
        half2_gains = {k: np.mean([all_gains[k][i] for i in half2_idx]) for k in K_GRID}
        
        # Compute shape metrics
        m1 = compute_shape_metrics_from_gains(half1_gains)
        m2 = compute_shape_metrics_from_gains(half2_gains)
        m1['mean_gain_128'] = half1_gains[128]
        m2['mean_gain_128'] = half2_gains[128]
        
        half1_metrics.append(m1)
        half2_metrics.append(m2)
    
    # Compute correlations
    for metric in split_half_correlations.keys():
        # Align by removing NaNs in either
        paired = [(m1[metric], m2[metric]) for m1, m2 in zip(half1_metrics, half2_metrics)
                  if not (np.isnan(m1.get(metric, np.nan)) or np.isnan(m2.get(metric, np.nan)))]
        
        if len(paired) > 2:
            v1, v2 = zip(*paired)
            r, _ = stats.pearsonr(v1, v2)
            split_half_correlations[metric].append(r)

# Report
print("\n" + "-"*60)
print("PRIMARY SHAPE METRICS:")
print("-"*60)
print(f"\n{'Metric':<20} {'Mean r':>10} {'SD r':>10} {'Spearman-Brown':>15}")
print("-"*60)

reliability_summary = {}
primary_metrics = ['early_ratio', 'log_slope_local', 'auc_log_k', 'auc_linear_k']
secondary_metrics = ['mean_gain_128']

for metric in primary_metrics:
    rs = split_half_correlations[metric]
    if rs:
        mean_r = np.mean(rs)
        sd_r = np.std(rs)
        sb_reliability = (2 * mean_r) / (1 + mean_r) if mean_r > -1 else np.nan
        print(f"{metric:<20} {mean_r:>10.3f} {sd_r:>10.3f} {sb_reliability:>15.3f}")
        reliability_summary[metric] = {'mean_r': mean_r, 'sd_r': sd_r, 'spearman_brown': sb_reliability}
    else:
        print(f"{metric:<20} {'N/A':>10}")

print("\n" + "-"*60)
print("SECONDARY (raw gain level - tracks fluency):")
print("-"*60)

for metric in secondary_metrics:
    rs = split_half_correlations[metric]
    if rs:
        mean_r = np.mean(rs)
        sd_r = np.std(rs)
        sb_reliability = (2 * mean_r) / (1 + mean_r) if mean_r > -1 else np.nan
        print(f"{metric:<20} {mean_r:>10.3f} {sd_r:>10.3f} {sb_reliability:>15.3f}")
        reliability_summary[metric] = {'mean_r': mean_r, 'sd_r': sd_r, 'spearman_brown': sb_reliability}

print("\nSpearman-Brown: estimated reliability if using all 20 windows.")
print("Values > 0.7 suggest acceptable reliability.")

## 6. Group Analysis

In [ ]:
GROUP_ORDER = ['low', 'mid', 'high']
SCORE_COLORS = {'low': '#e74c3c', 'mid': '#f39c12', 'high': '#2ecc71'}

df['score_bin'] = pd.Categorical(df['score_bin'], categories=GROUP_ORDER, ordered=True)

In [ ]:
print("="*80)
print("GROUP MEANS: Shape Metrics by Score Bin")
print("="*80)
print("\nPRIMARY SHAPE METRICS (memory-use profile):")

summary_rows = []

# Primary shape metrics
for metric in ['early_ratio', 'log_slope_local', 'auc_log_k', 'auc_linear_k']:
    print(f"\n{metric}:")
    for score_bin in GROUP_ORDER:
        subset = df[df['score_bin'] == score_bin][metric].dropna()
        mean = subset.mean()
        sem = subset.sem()
        ci95 = 1.96 * sem
        print(f"  {score_bin}: {mean:.4f} +/- {ci95:.4f} (n={len(subset)})")
        
        summary_rows.append({
            'metric': metric,
            'score_bin': score_bin,
            'mean': mean,
            'sem': sem,
            'ci95': ci95,
            'n': len(subset),
            'is_primary': True
        })

print("\n" + "-"*55)
print("SECONDARY METRICS (fluency/stability):")

for metric in ['mean_gain_128', 'baseline_nll', 'stability_early_ratio']:
    print(f"\n{metric}:")
    for score_bin in GROUP_ORDER:
        subset = df[df['score_bin'] == score_bin][metric].dropna()
        mean = subset.mean()
        sem = subset.sem()
        ci95 = 1.96 * sem
        print(f"  {score_bin}: {mean:.4f} +/- {ci95:.4f} (n={len(subset)})")
        
        summary_rows.append({
            'metric': metric,
            'score_bin': score_bin,
            'mean': mean,
            'sem': sem,
            'ci95': ci95,
            'n': len(subset),
            'is_primary': False
        })

df_group_summary = pd.DataFrame(summary_rows)

In [ ]:
# Build gain curve summary by group
curve_rows = []

for score_bin in GROUP_ORDER:
    subset = df[df['score_bin'] == score_bin]
    for k in K_GRID:
        gain_col = f'mean_gain_{k}'
        nll_col = f'mean_nll_{k}'
        
        curve_rows.append({
            'score_bin': score_bin,
            'k': k,
            'gain_mean': subset[gain_col].mean(),
            'gain_sem': subset[gain_col].sem(),
            'gain_ci95': 1.96 * subset[gain_col].sem(),
            'nll_mean': subset[nll_col].mean(),
            'n': len(subset),
        })

df_curve = pd.DataFrame(curve_rows)

print("\nGroup curve summary (mean_gain_k):")
print(df_curve.pivot(index='k', columns='score_bin', values='gain_mean').round(4))

## 7. Statistical Tests (Controlling for Confounds)

In [ ]:
# Standardize controls
df['token_count_z'] = (df['token_count'] - df['token_count'].mean()) / df['token_count'].std()
df['baseline_nll_z'] = (df['baseline_nll'] - df['baseline_nll'].mean()) / df['baseline_nll'].std()

# Filter for valid shape metrics
df_valid = df.dropna(subset=['early_ratio', 'log_slope_local', 'auc_log_k', 'auc_linear_k']).copy()
print(f"Valid essays for regression: {len(df_valid)} / {len(df)}")

In [ ]:
print("="*80)
print("REGRESSION MODELS: Shape Metrics ~ Group + Controls")
print("="*80)
print("\nControlling for token_count (length) and baseline_nll (fluency).")
print("Shape metrics are the PRIMARY outcomes of interest.\n")

regression_results = {}

# Primary shape metrics first
print("="*80)
print("PRIMARY SHAPE METRICS (memory-use profile)")
print("="*80)

for metric in ['early_ratio', 'log_slope_local', 'auc_log_k', 'auc_linear_k']:
    print(f"\n{'='*60}")
    print(f"{metric.upper()}")
    print(f"{'='*60}")
    
    formula = f'{metric} ~ C(score_bin) + token_count_z + baseline_nll_z'
    model = smf.ols(formula, data=df_valid).fit()
    
    print(f"\nFormula: {formula}")
    print(f"R²: {model.rsquared:.4f}, Adj R²: {model.rsquared_adj:.4f}, n={int(model.nobs)}")
    
    print("\nCoefficients:")
    for param in model.params.index:
        coef = model.params[param]
        pval = model.pvalues[param]
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")
    
    regression_results[metric] = {
        'formula': formula,
        'model': model,
        'r2': model.rsquared,
        'is_primary': True
    }

# Secondary metrics
print("\n" + "="*80)
print("SECONDARY (raw gain - tracks fluency/length)")
print("="*80)

for metric in ['mean_gain_128']:
    print(f"\n{'='*60}")
    print(f"{metric.upper()}")
    print(f"{'='*60}")
    
    formula = f'{metric} ~ C(score_bin) + token_count_z + baseline_nll_z'
    model = smf.ols(formula, data=df_valid).fit()
    
    print(f"\nFormula: {formula}")
    print(f"R²: {model.rsquared:.4f}, Adj R²: {model.rsquared_adj:.4f}, n={int(model.nobs)}")
    
    print("\nCoefficients:")
    for param in model.params.index:
        coef = model.params[param]
        pval = model.pvalues[param]
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")
    
    regression_results[metric] = {
        'formula': formula,
        'model': model,
        'r2': model.rsquared,
        'is_primary': False
    }

In [ ]:
# Effect sizes summary table
print("\n" + "="*80)
print("SUMMARY TABLE: Group Effects on Shape Metrics")
print("="*80)
print("\nShape metrics are the PRIMARY outcomes (memory-use profile).")
print("These should differ between groups if writing quality relates to context use.\n")

print(f"{'Metric':<20} {'Type':<10} {'mid vs low':>15} {'high vs low':>15} {'R²':>10}")
print("-"*75)

for metric, res in regression_results.items():
    model = res['model']
    metric_type = "PRIMARY" if res.get('is_primary', False) else "secondary"
    
    mid_coef = model.params.get('C(score_bin)[T.mid]', np.nan)
    mid_p = model.pvalues.get('C(score_bin)[T.mid]', 1.0)
    mid_sig = "***" if mid_p < 0.001 else "**" if mid_p < 0.01 else "*" if mid_p < 0.05 else ""
    
    high_coef = model.params.get('C(score_bin)[T.high]', np.nan)
    high_p = model.pvalues.get('C(score_bin)[T.high]', 1.0)
    high_sig = "***" if high_p < 0.001 else "**" if high_p < 0.01 else "*" if high_p < 0.05 else ""
    
    mid_str = f"{mid_coef:+.4f}{mid_sig}"
    high_str = f"{high_coef:+.4f}{high_sig}"
    
    print(f"{metric:<20} {metric_type:<10} {mid_str:>15} {high_str:>15} {res['r2']:>10.4f}")

print("\n* p<0.05, ** p<0.01, *** p<0.001")
print("\nInterpretation:")
print("  - Positive coefficients mean higher score_bin essays have higher metric values")
print("  - PRIMARY shape metrics show if groups differ in HOW they use context")
print("  - Secondary (mean_gain_128) tracks overall fluency/length effects")

## 8. Visualizations

In [ ]:
# Create output directory
output_dir = Path(OUTPUT_BASE) / EXPERIMENT
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

In [ ]:
# PRIMARY FIGURE: Gain curves by score bin
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Linear x-axis
ax = axes[0]
for score_bin in GROUP_ORDER:
    subset = df_curve[df_curve['score_bin'] == score_bin]
    ax.errorbar(subset['k'], subset['gain_mean'], yerr=subset['gain_ci95'],
                marker='o', capsize=3, label=score_bin, color=SCORE_COLORS[score_bin],
                linewidth=2, markersize=6)

ax.set_xlabel('Context length k (tokens)', fontsize=11)
ax.set_ylabel('Mean Gain = NLL(k=0) − NLL(k)', fontsize=11)
ax.set_title('Local Memory Curve by Score Bin\n(linear scale)', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

# Right: Log x-axis
ax = axes[1]
for score_bin in GROUP_ORDER:
    subset = df_curve[df_curve['score_bin'] == score_bin]
    subset_nonzero = subset[subset['k'] > 0]
    ax.errorbar(subset_nonzero['k'], subset_nonzero['gain_mean'], yerr=subset_nonzero['gain_ci95'],
                marker='o', capsize=3, label=score_bin, color=SCORE_COLORS[score_bin],
                linewidth=2, markersize=6)

ax.set_xscale('log')
ax.set_xlabel('Context length k (tokens, log scale)', fontsize=11)
ax.set_ylabel('Mean Gain = NLL(k=0) − NLL(k)', fontsize=11)
ax.set_title('Local Memory Curve by Score Bin\n(log scale)', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig(output_dir / 'gain_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Shape metrics distributions by group (PRIMARY OUTPUTS)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = ['early_ratio', 'log_slope_local', 'auc_log_k', 'auc_linear_k']
titles = [
    'Early Ratio (gain_16/gain_128)\n[PRIMARY]', 
    'Log Slope\n[PRIMARY]', 
    'AUC Log-k (emphasizes early saturation)\n[PRIMARY]', 
    'AUC Linear-k (emphasizes longer k)\n[PRIMARY]'
]

for idx, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    ax = axes[idx // 2, idx % 2]
    
    data = [df[df['score_bin'] == sb][metric].dropna() for sb in GROUP_ORDER]
    parts = ax.violinplot(data, positions=range(len(GROUP_ORDER)), showmeans=True, showmedians=True)
    
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(SCORE_COLORS[GROUP_ORDER[i]])
        pc.set_alpha(0.7)
    
    ax.set_xticks(range(len(GROUP_ORDER)))
    ax.set_xticklabels(GROUP_ORDER)
    ax.set_xlabel('Score Bin', fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Shape Metrics (Primary Outputs) by Score Bin', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(output_dir / 'shape_metrics_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Independence from fluency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ['early_ratio', 'log_slope_local']):
    for score_bin in GROUP_ORDER:
        subset = df_valid[df_valid['score_bin'] == score_bin]
        ax.scatter(subset['baseline_nll'], subset[metric], 
                   alpha=0.5, label=score_bin, color=SCORE_COLORS[score_bin], s=30)
    
    ax.set_xlabel('Baseline NLL (fluency proxy)', fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f'{metric} vs Fluency', fontsize=12, fontweight='bold')
    ax.legend(title='Score Bin')
    ax.grid(True, alpha=0.3)
    
    r, p = stats.pearsonr(df_valid['baseline_nll'].dropna(), df_valid[metric].dropna())
    ax.text(0.05, 0.95, f'r = {r:.3f}, p = {p:.4f}', transform=ax.transAxes, 
            fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.savefig(output_dir / 'shape_vs_fluency.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Reliability plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Within-essay stability
ax = axes[0]
data = [df[df['score_bin'] == sb]['stability_early_ratio'].dropna() for sb in GROUP_ORDER]
parts = ax.violinplot(data, positions=range(len(GROUP_ORDER)), showmeans=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(SCORE_COLORS[GROUP_ORDER[i]])
    pc.set_alpha(0.7)

ax.set_xticks(range(len(GROUP_ORDER)))
ax.set_xticklabels(GROUP_ORDER)
ax.set_xlabel('Score Bin', fontsize=11)
ax.set_ylabel('Within-essay SD of early_ratio', fontsize=11)
ax.set_title('Within-Essay Stability by Group\n(lower = more consistent)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Split-half reliability summary (primary shape metrics)
ax = axes[1]
primary_metrics = ['early_ratio', 'log_slope_local', 'auc_log_k', 'auc_linear_k']
metrics_in_summary = [m for m in primary_metrics if m in reliability_summary]
sb_values = [reliability_summary[m]['spearman_brown'] for m in metrics_in_summary]
colors = ['#3498db'] * len(metrics_in_summary)

bars = ax.bar(range(len(metrics_in_summary)), sb_values, color=colors, alpha=0.7)
ax.axhline(0.7, color='red', linestyle='--', label='Acceptable threshold (0.7)')
ax.set_xticks(range(len(metrics_in_summary)))
ax.set_xticklabels(metrics_in_summary, rotation=45, ha='right')
ax.set_ylabel('Spearman-Brown Reliability', fontsize=11)
ax.set_title('Split-Half Reliability of Shape Metrics\n(estimated full-test reliability)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(output_dir / 'reliability.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Results

In [ ]:
# Save essay-level results
df.to_csv(output_dir / 'essay_level_results_local_lag.csv', index=False)

# Save window-level results
df_windows.to_csv(output_dir / 'window_level_results_local_lag.csv', index=False)

# Save group curve summary
df_curve.to_csv(output_dir / 'group_curve_summary.csv', index=False)

# Save group metrics summary
df_group_summary.to_csv(output_dir / 'group_metrics_summary.csv', index=False)

# Save regression results
with open(output_dir / 'regression_local_lag.txt', 'w') as f:
    f.write("LOCAL LAG CURVE REGRESSION ANALYSIS (SLIDING WINDOWS)\n")
    f.write("=" * 80 + "\n\n")
    
    f.write("PRIMARY OUTPUTS: SHAPE METRICS\n")
    f.write("-" * 40 + "\n")
    f.write("Shape metrics characterize the 'memory-use profile' - how quickly\n")
    f.write("predictive benefit saturates as context length increases.\n\n")
    f.write("  * early_ratio = gain_16 / gain_128 (saturation speed)\n")
    f.write("  * log_slope_local = slope of gain vs log(k)\n")
    f.write("  * auc_log_k = AUC on log scale (emphasizes early saturation) [PRIMARY]\n")
    f.write("  * auc_linear_k = AUC on linear scale (emphasizes longer k)\n\n")
    
    f.write("WHAT WE'RE MEASURING:\n")
    f.write("-" * 40 + "\n")
    f.write("  - NLL on true next tokens (actual words written)\n")
    f.write("  - gain_k = NLL(k=0) - NLL(k): how much k tokens of context help\n")
    f.write("  - Shape of gain_k curve = 'local memory curve'\n")
    f.write("\n" + "=" * 80 + "\n\n")
    
    f.write(f"Configuration:\n")
    f.write(f"  Cohort: persuade_score_long (50/50/50)\n")
    f.write(f"  Target length T: {TARGET_LENGTH} tokens\n")
    f.write(f"  Windows per essay: {N_WINDOWS}\n")
    f.write(f"  Min preceding context: {MIN_CONTEXT} tokens\n")
    f.write(f"  End buffer: {END_BUFFER} tokens\n")
    f.write(f"  Context grid k: {K_GRID}\n")
    f.write(f"  Essays analyzed: {len(df)}\n")
    f.write(f"  Random seed: {RANDOM_SEED}\n")
    f.write("\n" + "=" * 80 + "\n\n")
    
    f.write("SPLIT-HALF RELIABILITY (Shape Metrics):\n")
    f.write("-" * 40 + "\n")
    for metric in ['early_ratio', 'log_slope_local', 'auc_log_k', 'auc_linear_k']:
        if metric in reliability_summary:
            vals = reliability_summary[metric]
            f.write(f"  {metric}: r={vals['mean_r']:.3f}, Spearman-Brown={vals['spearman_brown']:.3f}\n")
    f.write("\n" + "=" * 80 + "\n\n")
    
    f.write("REGRESSION MODELS:\n")
    f.write("-" * 40 + "\n")
    f.write("Models: metric ~ C(score_bin) + token_count_z + baseline_nll_z\n")
    f.write("(controlling for essay length and baseline fluency)\n\n")
    
    # Primary shape metrics first
    f.write("=" * 60 + "\n")
    f.write("PRIMARY SHAPE METRICS\n")
    f.write("=" * 60 + "\n\n")
    
    for metric in ['early_ratio', 'log_slope_local', 'auc_log_k', 'auc_linear_k']:
        if metric in regression_results:
            res = regression_results[metric]
            f.write(f"{metric.upper()} MODEL:\n")
            f.write(f"Formula: {res['formula']}\n")
            f.write(res['model'].summary().as_text())
            f.write("\n\n")
    
    f.write("=" * 60 + "\n")
    f.write("SECONDARY (raw gain level)\n")
    f.write("=" * 60 + "\n\n")
    
    if 'mean_gain_128' in regression_results:
        res = regression_results['mean_gain_128']
        f.write("MEAN_GAIN_128 MODEL:\n")
        f.write(f"Formula: {res['formula']}\n")
        f.write(res['model'].summary().as_text())
        f.write("\n")

print(f"\nSaved to {output_dir}/")
print(f"  - essay_level_results_local_lag.csv ({len(df)} rows)")
print(f"  - window_level_results_local_lag.csv ({len(df_windows)} rows)")
print(f"  - group_curve_summary.csv")
print(f"  - group_metrics_summary.csv")
print(f"  - regression_local_lag.txt")
print(f"  - gain_curves.png")
print(f"  - shape_metrics_distribution.png")
print(f"  - shape_vs_fluency.png")
print(f"  - reliability.png")

In [ ]:
# Final summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print(f"""
WHAT WE MEASURED:
  - For each window: NLL on the true next {TARGET_LENGTH} tokens
  - gain_k = NLL(k=0) - NLL(k): how much k preceding tokens help
  - Shape metrics averaged across {N_WINDOWS} windows per essay

PRIMARY OUTPUTS: SHAPE METRICS
  These characterize the 'memory-use profile' independent of fluency:
  
  * early_ratio = gain_16 / gain_128 (saturation speed)
  * log_slope_local = slope of gain vs log(k)
  * auc_log_k = AUC on log scale (emphasizes early saturation) [PRIMARY]
  * auc_linear_k = AUC on linear scale (emphasizes longer k)

RELIABILITY:
  * Split-half correlations with Spearman-Brown correction
  * Within-essay stability across {N_WINDOWS} windows

KEY QUESTION:
  Do groups differ in SHAPE of the local memory curve?
  (i.e., how they USE context, not just overall fluency)
  
  Shape metric differences = different memory-use profiles
  Raw gain differences = fluency/length effects
""")